# Extension Plan: Layers-as-Lenses Narrative for ReLU MLPs

This notebook is the render-friendly copy of the ReLU extension plan from `notes/research_spec.md`.
Use this file for viewing equations in Cursor.

---

## A. Recap of the Paper's Core Narrative

The paper makes three durable claims that we want to extend to ReLU MLPs:

- **Lens-Blinder-Residue decomposition.** For DLGN, the gradient on neuron $\mathbf{w}_k$ factors as Residue x Lens x Blinder x $y^* a \mathbf{x}$. The Lens $\Lambda_k(\mathbf{x})=\prod_{\kappa\ne k}\phi(\mathbf{w}_\kappa^\top\mathbf{x})$ comes from other neurons; the Blinder $\phi'(\mathbf{w}_k^\top\mathbf{x})$ comes from the neuron itself.
- **Dual role of a neuron and positive feedback.** $\mathbf{w}_k$ is simultaneously (a) a linear separator on data importance-weighted by its lens and (b) a component of lens for everyone else. This dual role is the engine of feature learning.
- **Hierarchy-seeking + self-opposing second-order dynamics.** These give leaf-first then root then descendants feature emergence, but also produce backlash along spurious directions.

The narrative-level claim is broader than DLGN: this should be a generic phenomenon for deep nonlinear architectures. The goal here is to make this claim legible in a standard ReLU MLP.

## B. Conceptual Translation: DLGN -> ReLU MLP

### B.1 Gates

- **DLGN-SF gate at layer $\ell$:** $\phi(\mathbf{w}_\ell^\top \mathbf{x})$ (a shallow halfspace in input space).
- **ReLU gate at layer $\ell$:** $\mathbb{1}[(W_\ell h_{\ell-1}(\mathbf{x}))_i > 0]$ (a deep halfspace).

For fixed $\mathbf{x}$ in one linear region, the effective input-space normal is:

$$
\tilde{\mathbf{w}}_{\ell,i}(\mathbf{x})
= \big((W_\ell)_{i,:}\,\mathrm{diag}(\mathbf{m}_{\ell-1}(\mathbf{x}))\,W_{\ell-1}\cdots \mathrm{diag}(\mathbf{m}_1(\mathbf{x}))\,W_1\big)^\top
$$

- For first-layer neurons, $\tilde{\mathbf{w}}_{1,i} = (W_1)_{i,:}$.
- For deeper neurons, $\tilde{\mathbf{w}}_{\ell,i}(\mathbf{x})$ is input-dependent.

### B.2 Paths

- **DLGN path:** $\pi = (m_1,\ldots,m_L) \in [M]^L$ with multiplicative gate product.
- **ReLU path:** same index $\pi$, but gated multilinear contribution:

$$
f(\mathbf{x})
= \sum_{\pi} \Bigl[\prod_{\ell=1}^{L-1} m_{\ell,i_\ell}(\mathbf{x})\Bigr]\cdot a_\pi \cdot (W_1)_{i_1,:}^\top \mathbf{x}
$$

with

$$
a_\pi = w_{L,i_{L-1}}\prod_{\ell=2}^{L-1}(W_\ell)_{i_\ell,i_{\ell-1}}.
$$

Each ReLU path has an activation indicator, a scalar coefficient $a_\pi$, and a separator $(W_1)_{i_1,:}$.

### B.3 Lens-Blinder for ReLU

The gradient w.r.t. $(W_k)_{i,:}$ has the same factorized structure:

$$
\nabla_{(W_k)_{i,:}} \mathcal{L}
= \mathbb{E}\Bigl[\underbrace{\sigma(-y^*\hat y)}_{\mathrm{Residue}}\cdot
\underbrace{\Lambda^{\mathrm{ReLU}}_{k,i}(\mathbf{x})}_{\mathrm{Lens}}\cdot
\underbrace{\mathbb{1}[(W_k h_{k-1})_i > 0]}_{\mathrm{Blinder}}\cdot
y^* h_{k-1}(\mathbf{x})^\top\Bigr].
$$

In ReLU, the lens is path-summed sensitivity through active paths passing via neuron $(k,i)$.

## C. Hypotheses

### Tier A (first-layer direct analogs)

- **H1:** First-layer ReLU neurons align with ODT vectors; large-norm neurons have high $|\cos|$ to some $\mathbf{u}_i$.
- **H2:** Hierarchical emergence appears (leaves first, then root, then children).
- **H3:** Capture epoch ordering reflects feature importance/scope.

### Tier B (path-level structure)

- **H4:** Active paths specialize toward ODT root-to-leaf paths.
- **H5:** Path coefficient sparsity emerges (lottery-ticket-like concentration).
- **H6:** Top-$k$ paths cover input space and align with leaf partition.

### Tier C (lens-blinder-residue empirics)

- **H7:** Near init, lens dominates gradient direction.
- **H8:** Lens reorients from diffuse to leaf-concentrated support.
- **H9:** Hierarchy-seeking analog via empirical second-order perturbation.
- **H10:** Self-opposing analog for spurious perturbation directions.

### Tier D (interventions/predictions)

- **H11:** Freezing deeper layers disrupts first-layer alignment.
- **H12:** Width scaling and pruning preserve aligned subnetworks.
- **H13:** Init-based heuristic predicts eventual lock-on directions.

### Tier E (deeper-layer gate analysis)

- **H14:** Effective deeper-layer gates align with ODT directions.
- **H15:** Layer roles specialize (early separators vs later lens modulators).

## D. Visualisations

- **V1:** First-layer norm vs $|\cos|$ scatter, colored by ODT level.
- **V2:** Per-neuron trajectories (norm and $|\cos|$ vs epoch).
- **V3:** ODT capture timeline (Gantt-style).
- **V4:** Path frequency + path entropy over epochs.
- **V5:** Lens-weighted data projections for chosen neurons.
- **V6:** Effective gate scatter for deeper layers.
- **V7:** Second-order probe heatmap.
- **V8:** Pruning curves vs accuracy.
- **V9:** Init-to-solution confusion matrix.

## E. Suggested Sequencing

1. Solidify first-layer narrative (H1-H3, V1-V3).
2. Add path-level analysis (H4-H6, V4/V8).
3. Lens/blinder empirics (H7-H8, V5).
4. Second-order probes (H9-H10, V7).
5. Interventions and prediction (H11-H13).
6. Deeper-layer effective gates (H14-H15).

## F. Pitfalls

- Linear-region explosion in ReLU.
- Sign ambiguity in cosine metrics.
- Bias terms can shift boundaries.
- Permutation invariance complicates neuron identity tracking.
- Path identifiability/degeneracy.
- Successful vs unsuccessful seed behavior should both be studied.

## G. Open Decisions

- Define ReLU paths as first-class code objects or analysis-only objects?
- Which subset of H1-H6 to prioritize first?
- Run parallel `bias=False` experiments for cleaner alignment analysis?
